# Entrega 4: Segmentación y Cálculos Analíticos

En esta etapa agregaremos una capa de inteligencia de negocios a nuestro Esquema Estrella.
Calcularemos nuevas métricas derivadas (Tasa de Ahorro) y crearemos una nueva dimensión de segmentación (Perfil de Ahorro) que servirá como clúster analítico en Tableau.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Carga de la Tabla de Hechos
Solo necesitamos cargar la tabla central (`fact_hogares`), ya que ahí residen los gastos e ingresos sobre los cuales calcularemos las nuevas métricas.

In [10]:
fact_hogares = pd.read_csv('../Data/modelo/esquema_estrella/fact_hogares.csv')
print(f"Filas iniciales: {fact_hogares.shape[0]}")
display(fact_hogares.head(3))

Filas iniciales: 33691


,ID_HOGAR,ID_GEOGRAFIA,MES_NUM,ID_POBREZA,MIEPERHO,FACTOR07,INGHOG2D,GASHOG2D,BRECHA_HOG,ING_PERCAPITA,GAS_PERCAPITA,BRECHA_PERCAPITA,LOG_INGHOG2D,LOG_GASHOG2D,LOG_BRECHA_PERCAP,GRU11HD,GRU21HD,GRU31HD,GRU41HD,GRU51HD,GRU61HD,GRU71HD,GRU81HD,GRU11HD_PCT,GRU21HD_PCT,GRU31HD_PCT,GRU41HD_PCT,GRU51HD_PCT,GRU61HD_PCT,GRU71HD_PCT,GRU81HD_PCT,TASA_AHORRO,ID_SEGMENTO
0,1,1,1,1,2,79.816757,52162.609375,34188.218750,17974.390625,26081.304688,17094.109375,8987.195312,10.862140,10.439666,9.103667,3845.363770,874.391602,1006.782715,1307.083496,3339.000000,3247.260742,1318.787598,3219.265869,0.2118,0.0482,0.0554,0.0720,0.1839,0.1788,0.0726,0.1773,0.344584,1
1,2,1,1,1,3,79.816757,40832.042969,40164.945312,667.097656,13610.680990,13388.315104,222.365885,10.617247,10.600775,5.408811,12792.489258,2384.777832,916.000000,665.926880,2408.242676,2768.000000,895.084717,752.953064,0.5424,0.1011,0.0388,0.0282,0.1021,0.1174,0.0380,0.0319,0.016338,2
2,3,1,1,1,1,79.816757,15098.497070,12308.838867,2789.658203,15098.497070,12308.838867,2789.658203,9.622417,9.418154,7.934033,2150.485107,603.583740,2589.000000,581.172546,2047.000000,233.000000,41.478962,557.595886,0.2443,0.0686,0.2941,0.0660,0.2325,0.0265,0.0047,0.0633,0.184764,1


## 1.5 Corrección de Participaciones de Gasto (`_PCT`)
En entregas previas, los porcentajes se calcularon sobre el Gasto Bruto (`GASHOG2D`). Sin embargo, este incluye gastos no monetarios (como el alquiler imputado) que no forman parte de los 8 rubros de la canasta básica.
Para garantizar que los porcentajes sumen exactamente 1.0 (100%) y sirvan para apilar barras en Tableau, vamos a recalcularlos usando como base la suma estricta de los 8 rubros monetarios.

In [11]:
grupos = [f'GRU{i}1HD' for i in range(1, 9)]
cols_pct = [f'{c}_PCT' for c in grupos]

# Suma estricta de los 8 rubros
gasto_canasta = fact_hogares[grupos].sum(axis=1)

# Recálculo
for col, pct_col in zip(grupos, cols_pct):
    fact_hogares[pct_col] = (fact_hogares[col] / gasto_canasta.replace(0, np.nan)).fillna(0).round(4)

print("Suma promedio de porcentajes recalculados:", fact_hogares[cols_pct].sum(axis=1).mean())


Suma promedio de porcentajes recalculados: 0.9994649283191359


## 2. Creación de Métricas Derivadas
Vamos a crear la métrica **TASA_AHORRO**. 
La fórmula es: `(INGHOG2D - GASHOG2D) / INGHOG2D`. 
Esta métrica relativa es mucho más poderosa que la brecha absoluta, ya que nos dice qué porcentaje de su sueldo logra ahorrar un hogar (o cuánto se está endeudando).

In [12]:
# Prevenir división por cero si algún ingreso es 0
ingreso_seguro = fact_hogares['INGHOG2D'].replace(0, np.nan)

# Cálculo de la Tasa de Ahorro
fact_hogares['TASA_AHORRO'] = (fact_hogares['INGHOG2D'] - fact_hogares['GASHOG2D']) / ingreso_seguro

# Si hubo división por cero, llenamos con 0
fact_hogares['TASA_AHORRO'] = fact_hogares['TASA_AHORRO'].fillna(0)

display(fact_hogares[['ID_HOGAR', 'INGHOG2D', 'GASHOG2D', 'BRECHA_HOG', 'TASA_AHORRO']].head(5))

,ID_HOGAR,INGHOG2D,GASHOG2D,BRECHA_HOG,TASA_AHORRO
0,1,52162.609375,34188.218750,17974.390625,0.344584
1,2,40832.042969,40164.945312,667.097656,0.016338
2,3,15098.497070,12308.838867,2789.658203,0.184764
3,4,41082.953125,30316.724609,10766.228516,0.262061
4,5,47659.160156,33076.910156,14582.250000,0.305970


## 3. Creación de la Dimensión de Segmentación (Clústeres)
Para responder a la pregunta analítica del proyecto, crearemos perfiles de hogares según su capacidad de ahorro.

Generamos un diccionario (Tabla de Dimensión) independiente para no duplicar cadenas de texto en la tabla de hechos.

In [13]:
# Definimos los perfiles analíticos y asignamos una paleta semántica (COLOR_HEX)
segmentos = [
    {'ID_SEGMENTO': 1, 'SEGMENTO': 'Ahorrador Sólido', 'DESCRIPCION': 'Ahorra 15% o más de su ingreso.', 'COLOR_HEX': '#2CA02C'},
    {'ID_SEGMENTO': 2, 'SEGMENTO': 'Equilibrio / Supervivencia', 'DESCRIPCION': 'Ahorra menos del 15%, pero no tiene déficit.', 'COLOR_HEX': '#FF7F0E'},
    {'ID_SEGMENTO': 3, 'SEGMENTO': 'Déficit Leve', 'DESCRIPCION': 'Gasta hasta un 15% más de lo que ingresa.', 'COLOR_HEX': '#D62728'},
    {'ID_SEGMENTO': 4, 'SEGMENTO': 'Déficit Crítico', 'DESCRIPCION': 'Gasta más de un 15% por encima de su ingreso.', 'COLOR_HEX': '#8C564B'}
]

dim_segmento = pd.DataFrame(segmentos)
display(dim_segmento)

,ID_SEGMENTO,SEGMENTO,DESCRIPCION,COLOR_HEX
0,1,Ahorrador Sólido,Ahorra 15% o más de su ingreso.,#2CA02C
1,2,Equilibrio / Supervivencia,"Ahorra menos del 15%, pero no tiene déficit.",#FF7F0E
2,3,Déficit Leve,Gasta hasta un 15% más de lo que ingresa.,#D62728
3,4,Déficit Crítico,Gasta más de un 15% por encima de su ingreso.,#8C564B


## 4. Asignación del Segmento a la Tabla de Hechos
Mapearemos el `ID_SEGMENTO` a cada hogar utilizando la `TASA_AHORRO`.

In [14]:
def clasificar_segmento(tasa):
    if tasa >= 0.15: return 1
    elif tasa >= 0: return 2
    elif tasa >= -0.15: return 3
    else: return 4

fact_hogares['ID_SEGMENTO'] = fact_hogares['TASA_AHORRO'].apply(clasificar_segmento)

print("Distribución de hogares por segmento:")
print(fact_hogares['ID_SEGMENTO'].value_counts().sort_index())

Distribución de hogares por segmento:
ID_SEGMENTO
1    14375
2     6735
3     4572
4     8009
Name: count, dtype: int64


## 5. Validación de Calidad de Datos (Tableau Ready)
**Criterio CRÍTICO:** Para que las tablas conecten limpiamente en Tableau sin reprocesamiento manual, garantizaremos dos cosas:
1. Cero valores Nulos (NaN) en las llaves foráneas.
2. Las llaves foráneas deben ser de tipo Entero (Int).

In [15]:
# Validación 1: Verificar nulos
nulos_en_llaves = fact_hogares[['ID_HOGAR', 'ID_GEOGRAFIA', 'ID_POBREZA', 'MES_NUM', 'ID_SEGMENTO']].isnull().sum()
assert nulos_en_llaves.sum() == 0, "ERROR: Se encontraron valores nulos en las llaves foráneas."
print("Validación 1 pasada: Cero valores nulos en llaves foráneas.")

# Validación 2: Casting de tipos
fact_hogares['ID_SEGMENTO'] = fact_hogares['ID_SEGMENTO'].astype(int)
print("Validación 2 pasada: Casting a Integer verificado.")

Validación 1 pasada: Cero valores nulos en llaves foráneas.
Validación 2 pasada: Casting a Integer verificado.


## 6. Exportación de Fuentes Finales
Exportamos el `fact_hogares.csv` actualizado y la nueva dimensión `dim_segmento.csv` a la carpeta del Esquema Estrella.

In [16]:
fact_hogares.to_csv('../Data/modelo/esquema_estrella/fact_hogares.csv', index=False, encoding='utf-8-sig')
dim_segmento.to_csv('../Data/modelo/esquema_estrella/dim_segmento.csv', index=False, encoding='utf-8-sig')

# Exportar versión paralela con encabezados descriptivos en español para lectura directa
mapa_espanol = {
    'ID_HOGAR': 'Identificador_Hogar', 'ID_GEOGRAFIA': 'Identificador_Geografia', 'MES_NUM': 'Numero_Mes', 'ID_POBREZA': 'Identificador_Pobreza',
    'MIEPERHO': 'Total_Miembros_Hogar', 'FACTOR07': 'Factor_Expansion_Anual', 'INGHOG2D': 'Ingreso_Neto_Total_Hogar', 'GASHOG2D': 'Gasto_Total_Bruto_Hogar',
    'BRECHA_HOG': 'Brecha_Monetaria_Hogar', 'ING_PERCAPITA': 'Ingreso_Monetario_Per_Capita', 'GAS_PERCAPITA': 'Gasto_Monetario_Per_Capita', 'BRECHA_PERCAPITA': 'Brecha_Monetaria_Per_Capita',
    'LOG_INGHOG2D': 'Logaritmo_Ingreso_Hogar', 'LOG_GASHOG2D': 'Logaritmo_Gasto_Hogar', 'LOG_BRECHA_PERCAP': 'Logaritmo_Brecha_Per_Capita',
    'GRU11HD': 'Gasto_Alimentos', 'GRU21HD': 'Gasto_Vestido_Calzado', 'GRU31HD': 'Gasto_Vivienda_Combustible_Electricidad', 'GRU41HD': 'Gasto_Muebles_Enseres_Mantenimiento',
    'GRU51HD': 'Gasto_Salud_Servicios_Medicos', 'GRU61HD': 'Gasto_Transportes_Comunicaciones', 'GRU71HD': 'Gasto_Esparcimiento_Cultura_Ensenanza', 'GRU81HD': 'Gasto_Otros_Bienes_Servicios',
    'GRU11HD_PCT': 'Porcentaje_Gasto_Alimentos', 'GRU21HD_PCT': 'Porcentaje_Gasto_Vestido_Calzado', 'GRU31HD_PCT': 'Porcentaje_Gasto_Vivienda_Combustible_Electricidad',
    'GRU41HD_PCT': 'Porcentaje_Gasto_Muebles_Enseres_Mantenimiento', 'GRU51HD_PCT': 'Porcentaje_Gasto_Salud_Servicios_Medicos', 'GRU61HD_PCT': 'Porcentaje_Gasto_Transportes_Comunicaciones',
    'GRU71HD_PCT': 'Porcentaje_Gasto_Esparcimiento_Cultura_Ensenanza', 'GRU81HD_PCT': 'Porcentaje_Gasto_Otros_Bienes_Servicios',
    'TASA_AHORRO': 'Tasa_Ahorro_Hogar', 'ID_SEGMENTO': 'Identificador_Segmento_Financiero'
}
fact_hogares_espanol = fact_hogares.rename(columns=mapa_espanol)
fact_hogares_espanol.to_csv('../Data/modelo/esquema_estrella/fact_hogares_espanol.csv', index=False, encoding='utf-8-sig')

print("¡Exportación exitosa! Fuentes intactas (fact_hogares.csv, dim_segmento.csv) + versión en español (fact_hogares_espanol.csv).")

¡Exportación exitosa! Las fuentes están listas para ser cargadas en Tableau.
